<a href="https://colab.research.google.com/github/peternguyendev/ai_lab/blob/main/lab51_multiagent/multiagent_lg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
!pip install -q langgraph matplotlib

In [49]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

"""
A) Define WorkflowState class contains states for the workflow
B) Define agents:
  1) planner_agent
  2) researcher_agent
  3) writer_agent
In the realword, are these functions will be replaced with actual agents?
Can you suggest the examples for these agents?
"""

class WorkflowState(TypedDict):
  task: str
  protocol: Literal['one_way', 'two_way', 'brainstorm']
  planner_notes: str
  researcher_notes: str
  writer_output: str
  message_log: list[str]

print('Import and state ready')

task = 'Create a 5-step launch plan for a new AI tutoring product for colledge students.'

agent_roles = [
      {'name': 'planner', 'responsibility': 'Define launch phases and required milestones.'},
      {'name': 'Researcher', 'responsibility': 'Surface market assumptions, risks, and evidence.'},
      {'name': 'write', 'responsibility': 'Conver inputs inot an actionable and concise launch plan.'},
  ]

print('Agent count:', len(agent_roles))
for role in agent_roles:
    print('-', role['name'], '=>', role['responsibility'])

def planner_agent(state: WorkflowState) -> WorkflowState:
  planner_notes = (
      '1) Define target segments on campus; '
      '2) Run a plilot with 2 deparments; '
      '3) Launch referral and ambassador channels.'
  )
  state['planner_notes'] = planner_notes
  state['log'].append(f'planner -> researcher: {planner_notes}')
  return state

def researcher_agent(state: WorkflowState) -> WorkflowState:
  base_notes = (
      'Key risks: student retention after week 2, privacy concerns, and faculty adoption lag. '
      'Evidence needed: pilot completion rate, week-4 retention, and NPS by major.'
  )
  if state['protocol'] == 'one_way':
    researcher_notes = base_notes
  elif state['protocol'] == 'two_way':
    researcher_notes = base_notes + 'Clarifying question to planner: prioritize growth vs retention for phase 1?'
    state['message_log'].append('researcher -> planner: Should phase 1 optimize growth or retention?')
    state['message_log'].append('planner -> researcher: Prioritize retention to validate product value before scale.')
  else: # brainstorm
    researcher_notes = (base_notes + ' Brainstromed angles: '
    'A0 facultiy-led onboarding, B) peer studygroup integrations, C) exam-season challenge campaigns.')

    state['message_log'].append('researcher brainstorm: generated 3 alternative go-to-market angles')

  state['message_log'].append(f"researcher -> writer: {researcher_notes}")
  return state

def writer_agent(state: WorkflowState) -> WorkflowState:
  writer_output = f"""Final 5-Step Launch Plan
  1. Segment: Identify priority student cohords and departments.
  2. Pilot: Run department-level pilot with clean success metrics.
  3. Validate: Track retention, NPS, and completion against risk assumptions.
  4. Expand: Ue ambassadors and referrals where pilot outcomes are strongest.
  5. Institutionalize: Package facutltiy onboarding and privacy/compiance guidance.

  Planner input: {state['planner_notes']}
  Research input: {state['researcher_notes']}
  Protocol used: {state['protocol']}
  """
  state['writer_output'] = writer_output
  state['message_log'].append('writer: produced final output')
  return state

  print(' Agent functions implemented')


Import and state ready
Agent count: 3
- planner => Define launch phases and required milestones.
- Researcher => Surface market assumptions, risks, and evidence.
- write => Conver inputs inot an actionable and concise launch plan.


In [50]:
# Connect to Langgraph
# Generate Langgraph nodes and edges
# Compile Langgraph application

def build_graph():
  graph = StateGraph(WorkflowState)
  graph.add_node('planner',planner_agent)
  graph.add_node('researcher', researcher_agent)
  graph.add_node('writer', writer_agent)

  graph.add_edge(START, 'planner')
  graph.add_edge('planner', 'researcher')
  graph.add_edge('researcher', 'writer')
  graph.add_edge('writer', END)

  return graph.compile()

compiled_graph = build_graph()
print('Graph compiled')


Graph compiled


In [51]:
# Define functions to run the langgraph

def run_protocol(protocol: str):
  initial_state: WorkflowState = {
      'task': task,
      'protocol': protocol,
      'planner_notes': '',
      'researcher_notes': '',
      'writer_output': '',
      'message_log': []
      }
  compiled_graph = build_graph
  return compiled_graph.invoke(initial_state)

results = {}
for protocol in ['one_way', 'two_way', 'brainstorm']:
  results[protocol] = run_protocol(protocol)
  print(f'\n=== {protocol.upper()}===')
  print(results[protocol]['writer_output'])
  print('Message: ')
  for message in results[protocol]['message_log']:
    print(' ', message)


AttributeError: 'function' object has no attribute 'invoke'

In [ ]:
# Protocol comparison visualization

import matplotlib.pyplot as plt

protocols = ['one_way', 'two_way', 'brainstorm']
colors = ['#4C9BE8', '#F5A623', '#7ED321']
message_counts = [len(results[p]['message_log']) for p in protocols]
back_and_forth = {'one_way': 0, 'two_way': 1, 'brainstorm': 2}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Module 5 Solution - Part 1 Protocol Comparison", fontsize=13, fontweight='bold', y=1.02)

ax1 = axes[0]
print(f'ax1 y message_counts = {message_counts}')
bars1 = ax1.bar(protocols, message_counts, color=colors, width=0.5)
ax1.set_title("Total Messages Exchanged", fontweight='bold')
ax1.set_ylabel("Message count")
ax1.set_ylim(0, max(message_counts) + 2)
for bar, count in zip(bars1, message_counts):
  ax1.text(bar.get_x() + bar.get_width() /2, bar.get_height() + 0.1, str(count), ha='center', va='bottom', fontweight='bold')

ax2 = axes[1]
btf_vals = [back_and_forth[p] for p in protocols]
print(f'ax2 y btf_vals = {btf_vals}')
bars2 = ax2.bar(protocols, btf_vals, color=colors, width=0.5)
ax2.set_title('Coordination Loops (feedback Round-Trips)', fontweight='bold')
ax2.set_ylabel("Round trips")
ax2.set_ylim(0, max(btf_vals) + 1)
for bar, val in zip(bars2, btf_vals):
  ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05, str(val), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
example_protocol = 'two_way'
quick_state = run_protocol(example_protocol)

print(f'Protocol: {example_protocol}\n')
print("Final output: \n")
print(quick_state['writer_output'])
print("\nMessage exchanged: ")
for msg in quick_state['message_log']:
  print(" .", msg)
  print(f'Total messages: {len(quick_state['message_log'])}\n')

In [ ]:
# 1. New task and a customized 4-role agent team

custom_task = (
    'Design a 4-step roolout for an internal AI knowledge assistant '
    'serving a 300-person engineering organization, balancing security review'
    'cahnge management, and measureable productivity gains.'
)

custom_agent_roles = [
    {'name': 'planner', 'responsibility': 'Sequence rollout phases and milestone gates.'},
    {'name': 'researcher', 'responsibility': 'Surface adoption risks and eveidence needed to validate them,'},
    {'name': 'compliance', 'responsibility': 'Define data-handling guardrails and approval checkpoints (folded into researcher_agent for this graph).'},
    {'name': 'writer', 'responsibility': 'Synthesize input into an actionable rollout plan with named owners.'},
]

print('Customized agent count:', len(custom_agent_roles))
for role in custom_agent_roles:
  print(' -', role['name'], '=>', role['responsibility'])

# 2. Run all three protocols on the new task using the existing compiled graph

custom_results = {}
for protocol in ['one_way', 'two_way', 'brainstorm']:
  custom_results[protocol] = compiled_graph.invoke({
        'task': custom_task,
        'protocol': protocol,
        'planner_notes': '',
        'researcher_notes': '',
        'writer_output': '',
        'message_log': [],
  })

# 3. Compare results accross protocols on the customized task
for protocol in ['one_way', 'two_way', 'brainstorm']:
  result = custom_results[protocol]
  print(f'\n=== Customtask - {protocol.upper()} ===')
  print(result['writer_output'])
  print(f" Total messages exchanged: {len(result['message_log'])}")

# 4. Observation: how protocol choice changed behavior on the new task
print('\n--- Observation ---')
print('Same agent functions, different task: brainstorm again surfaces alternative')
print('angles (faculty-led, peer-group, exam-season here adapted in concept to)')
print('rollout pillasrs, two_way adds a clarifying Q&A that catches asumptions')
print('about retention vs. growth priority, and one_way is fastest but inherits')
print('upstream choices unchallenge. For a high-stakes internal rollout where')
print('security review and change management dominate risk, two_way is the safer')
print('default; brainstorm is preferable in the early-discovery phase before')
print('phasing is locked in.')





In [ ]:
!pip install -q langgraph pandas

In [ ]:
# Part B: Control Nodes
# The graph will branch on tool errors and loop when quality is below target.

def coordinator(state: ControlState) -> ControlState:
  state['draft'] = (
      f"Draft plan for: {state['task']} | "
      'Steps: segment users, run plilot, measure retention, exppand channels, formalze operations.'
  )
  state['needs_tool'] = 'budget' in state['task'].lower() or 'cost' in state['task'].lower()
  state['log'].append('coordinator: created draft and evaluated tool need')
  return state

def tool_user(state: ControlState) -> ControlState:
  try:
    if state['needs_tool']:
      # Call all three registered tools through the MCP interface.
      budget = mcp_call('estimate_launch_budget', students=2000, tier='standard')
      support = mcp_call('estimate_support_load', students=2000)
      marketing = mcp_call('estimate_marketing_spend', student=2000)
      state['draft'] += (
          f" | Budget estimate: ${budget}"
          f" | Support specialists needed: {support}"
          f" | Marketing spend: ${marketing}"
      )
      state['log'].append('tool_user: skipped tools (not required)')
  except Exception as exc:
    state['error'] = str(exc)
    state['log'].append(f'tool_user error: {exc}')
  return state

def reviewer(state: ControlState) -> ControlState:
  quality = 0.52

  if state['protocol'] == 'two_way':
    quality += 0.12
  elif state['protocol'] == 'brainstorm':
    quality += 0.20

  if 'Budget estimate' in state['draft']:
    quality += 0.10
  if 'Support specialists needed' in state['draft']:
    quality += 0.06
  if 'Marketing spend' in state['draft']:
    quality += 0.04
  if 'KPI' in state['draft']:
    quality += 0.06
  if 'owner' in state['draft'].lower():
    quality += 0.08

  state['quality_score'] = min(quality, 0.99)
  state['feedback'] = (
      'Revise: add measurable KPIs, ownership, and timeline dependencies.'
      if state['quality_score'] < 0.78
      else 'Quality acceptable.'
  )
  state['log'].append(f"reviewer: quality={state['quality_score']:.2f} feedback={state['feedback']}")
  return state

def reviser(state: ControlState) -> ControlState:
  state['retries'] += 1
  state['draft'] += (
      f" | Revision {state['retries']}: Added KPI targets (activtion, week-4 retention), "
      'owner per step, and milestone timeline dependencies.'
  )
  state['log'].append(f'reviser: retry #{state['retries']}')
  return state

def finalize(state: ControlState) -> ControlState:
  state['final_output'] = state['draft']
  state['log'].append('finalize: produced final output')
  return state

def failover(state: ControlState) -> ControlState:
  state['final_output'] = f'FAILED: {state['error']}'
  state['log'].append('failover: returned fallback output')
  return state

print(' Nodes implemented')



In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
import pandas as pd

class ControlState(TypedDict):
  task: str
  protocol: Literal['one_way', 'two_way', 'brainstorm']
  draft: str
  feedback: str
  final_output: str
  quality_score: float
  retries: int
  max_retries: int
  needs_tool: bool
  error: str
  log: list[str]

print(' Imports and state ready')



In [ ]:
# ---- Replace the starter's flat $12/student model ------
# In production the rate card would come from a pricing serice or DB query;
# The dict below stands in for that lookup so the lab runs offline.

_BUDGET_RATE_CARD = {
    # tier -> (per-student cost in USD, volume-discount threshold in students)
    'self_serve': (8, 1500),
    'standard':   (12, 1000),
    'premium':    (20, 500),
}

def estimatej_launch_budget(students: int, tier: str = 'standard') -> int:
  if students <= 0:
    raise ValueError('students must be > 0')
  if tier not in _BUDGET_RATE_CARD:
    raise ValueError(f'Unknown tier {tier!r}; valid tiers: {list(_BUDGET_RATE_CARD)}')
  per_student, discount_at = _BUDGET_RATE_CARD[tier]
  if students >= discount_at:
    per_student = int(round(per_student * 0.9)) # 10% volume discount
  return students * per_student

def estimate_support_load(students: int) -> int:
  """Rough headcount of support specialists needed (1 per 80 students). """
  if students <= 0:
    raise ValueError('students must be > 0')
  return max(1, students // 80)

# --- Thrid tool added to the registry ----------------------
# Adding it requires onluy a new entry in TOOLS; tool_user keeps using mcp_call.

def estimate_marketing_spend(students: int, channel_mix: dict | None = None) -> int:
  """Estimate marketing spend for acquiring `students` learners.
  `channel_mix` is the share of qcquisition coming from each channel; only paid acquisition contributes to spend in this simplified model.
  """
  if students <= 0:
    raise ValueError('students must be > 0')
  channel_mix = channel_mix or {'paid': 0.6, 'organic': 0.4}
  paid_share = channel_mix('paid', 0.0)
  if not 0.0 <= paid_share <= 1.0:
    raise ValueError(f'paid_share must be in [0, 1] got {paid_share}')
  cost_per_acquired_student = 25 # blended CAC for paid channels
  return int(students * paid_share * cost_per_acquired_student)

TOOLS = {
    'estimate_launch_budget': estimatej_launch_budget,
    'estimate_support_load': estimate_support_load,
    'estimate_marketing_spend': estimate_marketing_spend,
}

def mcp_call(tool_name: str, **kwargs):
  tool = TOOLS.get(tool_name)
  if tool is None:
    raise ValueError(f'Unknown tool: {tool_name}')
  try:
    return tool(**kwargs)
  except TypeError as exc:
    raise ValueError(f'Invalud arguments for {tool_name}: {kwargs}') from exc

print(' MCP-style adapter ready (3 tools registered)')
print(' estimate_launch_budget(2000, tier="standard") =', mcp_call('estimate_launch_budget', students=2000, tier='standard'))
print(' estimate_launch_budget(2000, tier="premium") =', mcp_call('estimate_launch_budget',students=2000, tier='premium'))
print(' estimate_support_load = ', mcp_call('estimate_support_load', student=2000))
print(' estimate_marketing_spend(2000) =', mcp_call('estimate_marketing_spend', students=2000))


In [ ]:
# Part C: Conditional Routing + Iteration

# Routing policy:

# Tool error -> failover
# Quality >= threshold -> finalize
# Quality below threshold with retries left -> reviser loop

def route_after_tool(state: ControlState):
  return 'failover' if state['error'] else 'reviewer'

def route_after_review(state: ControlState):
  if state['quality_score'] >= 0.78:
    return 'finalize'
  if state['retries'] >= state['max_retries']:
    return 'finalize'
  return 'reviser'

def build_graph():
  graph = StateGraph(ControlState)
  graph.add_node('coordinator', coordinator)
  graph.add_node('tool_user', tool_user)
  graph.add_node('reviewer', reviewer)
  graph.add_node('reviser', reviser)
  graph.add_node('failover', failover)

  graph.add_edge(START, 'coordinator')
  graph.add_edge('coordinator', 'tool_user')
  graph.add_conditional_edges('tool_user', route_after_tool, {'reviewer': 'reviewer', 'failover': 'failover'})
  graph.add_conditional_edges('reviewer', route_after_review, {'reviser': 'reviser', 'finalize': 'finalize'})
  graph.add_edge('reviser', 'reviewer')
  graph.add_edge('finalize', END)
  graph.add_edge('failover', END)

  return graph.compile()





In [ ]:
# Part D: Benchmarking
# We compare protocol-level trade-offs across two task types.

def run_config(protocol: str, task_text: str, max_retries: int = 2):
  state: ControlState = {
      'task': task_text,
      'protocol': protocol,
      'draft': '',
      'final_output': '',
      'quality_score': 0.0,
      'retries': 0,
      'max_retries': max_retries,
      'needs_tool': False,
      'error': '',
      'log': []
  }

  result = build_graph.invoke(state)
  complexity_proxy = result['retries'] + len(result['log'])
  return {
      'protocol': protocol,
      'task': task_text,
      'quality_score': result['quality_score'],
      'retries': result['retries'],
      'log_teps': len(result['log']),
      'complexity_proxy': complexity_proxy
  }, result

tasks = [
    'Create a launch coordination plan with timeline and owners.',
    'Create a launch plan and include a budget estimate for 2000 students.'
]
protocols = ['one_way', 'two_way', 'brainstorm']

rows = []
example_outputs = {}
for task_text in tasks:
  for protocol in protocols:
    row, result = run_config(protocol, task_text, max_retries=2)
    rows.append(row)
    example_outputs[(protocol, task_text)] = result['final_output']

df = pd.DataFrame(rows)
display(df)

summary = df.groupby('protocol')[['quality_score', 'complexity_proxy', 'retries']].mean().reset_index()
print('\nAverage metrics by protocol: ')
display(summary)

print('\Sample output (brainstorm + budget task):')
print(example_outputs[('brainstorm', task[1])])
